In [1]:
from datetime import datetime
import json
from src.helpers import get_project_folders, normalize_project
from src.metric import TimestampMetric, MonthMetric, get_phase
import re
from pathlib import Path
from tqdm.contrib.concurrent import process_map

/home/jortvd/Documents/GitKraken/rust-migration-thesis/analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
INPUT_FOLDER = "/home/jortvd/thesis-data/github-results-25-3-2026"
OUTPUT_FOLDER = "../results"

In [3]:
def iterate_parallel(func, files: list[Path]) -> list:
    items = []
    for file in files:
        items.append((file.parent.name, file))
    results = process_map(func, items, chunksize=1)
    results = [result for result in results if result is not None]
    return [item for sublist in results if isinstance(sublist, list) for item in sublist] if results and isinstance(results[0], list) else results

In [4]:
metric = SumMetric("issue_count")

issue_files = sorted([f for f in Path(INPUT_FOLDER).glob("**/issue_*.json") if "_timeline" not in f.name and f.name.endswith(".json")])
def process_item(item):
    project_folder, issue_file = item

    with issue_file.open() as f:
        issue = json.load(f)

    return (
        normalize_project(project_folder),
        datetime.fromisoformat(issue["created_at"]),
        1,
        False
    )

metric.add_many(iterate_parallel(process_item, issue_files))
metric.save(OUTPUT_FOLDER)

NameError: name 'SumMetric' is not defined

In [5]:
BUG_TEXT_REGEX = re.compile(
    r'\b(error|bug|fix|issue|mistake|incorrect|fault|defect|flaw)\b', 
    re.IGNORECASE
)

BUG_LABEL_REGEX = re.compile(r'(bug|defect|fix|error)', re.IGNORECASE)

def is_defect(issue: dict) -> bool:
    labels = issue.get("labels", [])
    for label in labels:
        label_text = label.get("name", "") if isinstance(label, dict) else str(label)
        if BUG_LABEL_REGEX.search(label_text):
            return True

    title = issue.get("title", "")
    body = issue.get("body", "")
    text_to_search = f"{title} \n {body}"
    
    return bool(BUG_TEXT_REGEX.search(text_to_search))

In [6]:
metric = MonthMetric("defect_issue_count")

issue_files = sorted([f for f in Path(INPUT_FOLDER).glob("**/issue_*.json") if "_timeline" not in f.name and f.name.endswith(".json")])
def process_item(item):
    project_folder, issue_file = item

    with issue_file.open() as f:
        issue = json.load(f)

    if not is_defect(issue):
        return None

    return (
        normalize_project(project_folder),
        datetime.fromisoformat(issue["created_at"]),
        1,
        False
    )

metric.add_many(iterate_parallel(process_item, issue_files))
metric.save(OUTPUT_FOLDER)

100%|██████████| 57143/57143 [00:07<00:00, 7215.67it/s]


In [7]:
metric = TimestampMetric("defect_issue_resolution_time")

issue_files = sorted([f for f in Path(INPUT_FOLDER).glob("**/issue_*.json") if "_timeline" not in f.name and f.name.endswith(".json")])
def process_item(item):
    project_folder, issue_file = item

    with issue_file.open() as f:
        issue = json.load(f)

    if not is_defect(issue):
        return None

    with (issue_file.parent / f"{issue_file.stem}_timeline.json").open() as f:
        timeline = json.load(f)

    from_time = datetime.fromisoformat(issue["created_at"])
    to_time = None

    for event in timeline:
        time = datetime.fromisoformat(event["created_at"])
        if event.get("event") == "closed" and (to_time is None or time < to_time):
            to_time = time

    if to_time is None or get_phase(project_folder, from_time) != get_phase(project_folder, to_time):
        return None
    
    return (
        normalize_project(project_folder),
        from_time,
        (to_time - from_time).total_seconds() / (3600.0 * 24.0 * 30.0),
        False
    )

metric.add_many(iterate_parallel(process_item, issue_files))
metric.save(OUTPUT_FOLDER)

100%|██████████| 57143/57143 [00:09<00:00, 5986.28it/s]


In [ ]:
CRASH_KEYWORD_REGEX = re.compile(
    r'\b(crash|crashes|crashed|segfault|segmentation fault|panic|core dumped|fatal error|out of memory|oom)\b', 
    re.IGNORECASE
)

STACK_TRACE_REGEX = re.compile(
    r'(Traceback \(most recent call last\):|'         # Python
    r'\s+at [a-zA-Z0-9_$.]+\([a-zA-Z0-9_$.]+:\d+\)|'  # Java / JS / C#
    r'Exception in thread ".*"|'                      # Java
    r'panic: runtime error:|'                         # Go
    r'\[\s*\d+\.\d+\] \w+\[\d+\]: segfault at)',      # C / Linux Kernel
    re.IGNORECASE
)

CRASH_LABEL_REGEX = re.compile(r'(crash|fatal|panic|segfault|exception)', re.IGNORECASE)

def is_crash(issue: dict) -> bool:
    labels = issue.get("labels", [])
    for label in labels:
        label_text = label.get("name", "") if isinstance(label, dict) else str(label)
        if CRASH_LABEL_REGEX.search(label_text):
            return True

    title = issue.get("title", "")
    body = issue.get("body", "")
    
    if body and STACK_TRACE_REGEX.search(body):
        return True

    text_to_search = f"{title} \n {body}"
    if CRASH_KEYWORD_REGEX.search(text_to_search):
        return True
        
    return False

In [ ]:
metric = SumMetric("defect_issue_count")

for project_folder in get_project_folders(INPUT_FOLDER):
    for issue_file in sorted(project_folder.glob("issue_*.json")):
        if "timeline" in issue_file.name:
            continue
            
        with issue_file.open() as f:
            issue = json.load(f)

        if not is_defect(issue):
            continue

        metric.add(
            normalize_project(project_folder.name),
            datetime.fromisoformat(issue["created_at"]),
            1
        )

metric.save(OUTPUT_FOLDER)